In [187]:
from itertools import combinations

 ## Sudoku generator and solver - version 1 ##

In [214]:
sdk1 = '200709005600030002400000007002000400000104000000567000840000063029000810007000500'
sdk = '084520000000007000306908000200600800065000740008005002000106508000200000000089230'
def string_to_board(s):
    board = []
    for i in range(0,81,9):
        row = [int(x) for x in s[i:i+9]]
        board.append(row)
    return board


board = string_to_board(sdk)


In [215]:
def draw_board(board):
    for r in range(9):
        if r % 3 == 0 and r != 0:
            print("-"*21)

        row = ""
        for c in range(9):
            if c % 3 == 0 and c != 0:
                row += "| "

            val = board[r][c] if board[r][c] != 0 else "."
            row += str(val) + " "

        print(row)
draw_board(board)

. 8 4 | 5 2 . | . . . 
. . . | . . 7 | . . . 
3 . 6 | 9 . 8 | . . . 
---------------------
2 . . | 6 . . | 8 . . 
. 6 5 | . . . | 7 4 . 
. . 8 | . . 5 | . . 2 
---------------------
. . . | 1 . 6 | 5 . 8 
. . . | 2 . . | . . . 
. . . | . 8 9 | 2 3 . 


In [216]:
def get_candidates(board, row, col):
    if board[row][col] != 0:
        return set()

    digits = set(range(1, 10))

    row_nums = set(board[row])

    col_nums = {board[r][col] for r in range(9)}

    start_row = (row // 3) * 3
    start_col = (col // 3) * 3

    box_nums = {
        board[r][c]
        for r in range(start_row, start_row + 3)
        for c in range(start_col, start_col + 3)
    }

    used = row_nums | col_nums | box_nums
    used.discard(0)

    return digits - used

In [217]:
def get_all_candidates(board):
    candidates = [[set() for _ in range(9)] for _ in range(9)]

    for row in range(9):
        for col in range(9):
            if board[row][col] == 0:
                candidates[row][col] = get_candidates(board, row, col)

    return candidates


In [218]:
# Naked Singles

def naked_singles(board, candidates):
    progress = False

    for row in range(9):
        for col in range(9):
            if board[row][col] == 0 and len(candidates[row][col]) == 1:
                value = next(iter(candidates[row][col]))
                board[row][col] = value
                print(f'Populated {value} at {row},{col}')
                progress = True

    return progress

In [219]:
# Hidden Singles

def hidden_singles_rows(board, candidates):

    for row in range(9):
        digit_positions = {digit: [] for digit in range(1, 10)}

        for col in range(9):
            if board[row][col] == 0:
                for digit in candidates[row][col]:
                    digit_positions[digit].append((row, col))

        for digit in range(1,10):
            if len(digit_positions[digit]) == 1:
                r, c = digit_positions[digit][0]
                board[r][c] = digit
                print(f'Hidden ROW single {digit} placed at position {r, c}')
                return True
    return False

def hidden_singles_columns(board, candidates):

    for col in range(9):
        digit_positions = {digit: [] for digit in range(1, 10)}

        for row in range(9):
            if board[row][col] == 0:
                for digit in candidates[row][col]:
                    digit_positions[digit].append((row, col))

        for digit in range(1,10):
            if len(digit_positions[digit]) == 1:
                r, c = digit_positions[digit][0]
                board[r][c] = digit
                print(f'Hidden COLUMN single {digit} placed at position {r, c}')
                return True
    return False

def hidden_singles_boxes(board, candidates):

    for box_row_start in range(0,9,3):
        for box_col_start in range(0,9,3):
            digit_positions = {digit: [] for digit in range(1, 10)}

            for row in range(box_row_start,box_row_start+3):
                for col in range(box_col_start,box_col_start+3):
                    if board[row][col] == 0:
                        for digit in candidates[row][col]:
                            digit_positions[digit].append((row, col))

            for digit in range(1,10):
                if len(digit_positions[digit]) == 1:
                    r, c = digit_positions[digit][0]
                    board[r][c] = digit
                    print(f'Hidden BOX single {digit} placed at position {r, c}')
                    return True
    return False

In [220]:
# Helper Functions

def get_row_cells(row):
    return [(row, col) for col in range(9)]

def get_col_cells(col):
    return [(row, col) for row in range(9)]

def get_box_cells(start_row, start_col):
    return [
        (row, col)
        for row in range(start_row, start_row + 3)
        for col in range(start_col, start_col + 3)
    ]

In [221]:
# Naked Pairs

def naked_sets_in_unit(board, candidates, unit_cells, set_size):
    progress = False

    candidate_cells = [
        (row, col)
        for row, col in unit_cells
        if board[row][col] == 0 and 2 <= len(candidates[row][col]) <= set_size
    ]

    for cell_combo in combinations(candidate_cells, set_size):
        combo_digits = set()

        for row, col in cell_combo:
            combo_digits.update(candidates[row][col])

        if len(combo_digits) == set_size:
            combo_cells = set(cell_combo)

            for row, col in unit_cells:
                if (row, col) not in combo_cells and board[row][col] == 0:
                    before = candidates[row][col].copy()
                    candidates[row][col] -= combo_digits

                    if candidates[row][col] != before:
                        print(
                            f'Naked set {combo_digits} removed from {(row, col)} '
                            f'using cells {cell_combo}'
                        )
                        progress = True

            if progress:
                return True

    return False

def naked_pairs_rows(board, candidates):
    progress = False

    for row in range(9):
        if naked_sets_in_unit(board, candidates, get_row_cells(row), 2):
            progress = True

    return progress

def naked_pairs_cols(board, candidates):
    progress = False

    for col in range(9):
        if naked_sets_in_unit(board, candidates, get_col_cells(col), 2):
            progress = True

    return progress

def naked_pairs_boxes(board, candidates):
    progress = False

    for start_row in range(0, 9, 3):
        for start_col in range(0, 9, 3):
            if naked_sets_in_unit(board, candidates, get_box_cells(start_row, start_col), 2):
                progress = True

    return progress

def naked_pairs(board, candidates):
    if naked_pairs_rows(board, candidates):
        return True
    if naked_pairs_cols(board, candidates):
        return True
    if naked_pairs_boxes(board, candidates):
        return True
    return False

def naked_triples_rows(board, candidates):
    progress = False

    for row in range(9):
        if naked_sets_in_unit(board, candidates, get_row_cells(row), 3):
            progress = True

    return progress

def naked_triples_cols(board, candidates):
    progress = False

    for col in range(9):
        if naked_sets_in_unit(board, candidates, get_col_cells(col), 3):
            progress = True

    return progress

def naked_triples_boxes(board, candidates):
    progress = False

    for start_row in range(0, 9, 3):
        for start_col in range(0, 9, 3):
            if naked_sets_in_unit(board, candidates, get_box_cells(start_row, start_col), 3):
                progress = True

    return progress

def naked_triples(board, candidates):
    if naked_triples_rows(board, candidates):
        return True
    if naked_triples_cols(board, candidates):
        return True
    if naked_triples_boxes(board, candidates):
        return True
    return False

In [222]:
# Hidden Pairs


def hidden_sets_in_unit(board, candidates, unit_cells, set_size):
    progress = False
    digit_positions = {digit: [] for digit in range(1,10)}

    for row, col in unit_cells:
        if board[row][col] == 0:
            for digit in candidates[row][col]:
                digit_positions[digit].append((row, col))

    candidate_digits = [
        digit for digit in range(1, 10)
        if 1 <= len(digit_positions[digit]) <= set_size
    ]

    for digit_combo in combinations(candidate_digits, set_size):
        combo_set = set(digit_combo)
        combo_cells = set()

        for digit in digit_combo:
            combo_cells.update(digit_positions[digit])

        if len(combo_cells) == set_size:
            for row, col in combo_cells:
                before = candidates[row][col].copy()
                candidates[row][col] &= combo_set

                if candidates[row][col] != before:
                    print(
                        f'Hidden set {combo_set} reduced {(row, col)} '
                        f'from {before} to {candidates[row][col]}'
                    )
                    progress = True

            if progress:
                return True

    return False

def hidden_pairs_rows(board, candidates):
    progress = False

    for row in range(9):
        if hidden_sets_in_unit(board, candidates, get_row_cells(row), 2):
            progress = True

    return progress

def hidden_pairs_cols(board, candidates):
    progress = False

    for col in range(9):
        if hidden_sets_in_unit(board, candidates, get_col_cells(col), 2):
            progress = True

    return progress

def hidden_pairs_boxes(board, candidates):
    progress = False

    for start_row in range(0, 9, 3):
        for start_col in range(0, 9, 3):
            if hidden_sets_in_unit(board, candidates, get_box_cells(start_row, start_col), 2):
                progress = True

    return progress

def hidden_pairs(board, candidates):
    if hidden_pairs_rows(board, candidates):
        return True
    if hidden_pairs_cols(board, candidates):
        return True
    if hidden_pairs_boxes(board, candidates):
        return True
    return False

def hidden_triples_rows(board, candidates):
    progress = False

    for row in range(9):
        if hidden_sets_in_unit(board, candidates, get_row_cells(row), 3):
            progress = True

    return progress

def hidden_triples_cols(board, candidates):
    progress = False

    for col in range(9):
        if hidden_sets_in_unit(board, candidates, get_col_cells(col), 3):
            progress = True

    return progress

def hidden_triples_boxes(board, candidates):
    progress = False

    for start_row in range(0, 9, 3):
        for start_col in range(0, 9, 3):
            if hidden_sets_in_unit(board, candidates, get_box_cells(start_row, start_col), 3):
                progress = True

    return progress

def hidden_triples(board, candidates):
    if hidden_triples_rows(board, candidates):
        return True
    if hidden_triples_cols(board, candidates):
        return True
    if hidden_triples_boxes(board, candidates):
        return True

    return False

In [223]:
# Pointing Pairs/Triples

def pointing_pairs_triples(board, candidates):

    for r in range(0,9,3):
        for c in range(0,9,3):
            digit_positions = {digit: [] for digit in range(1,10)}

            for row in range(r, r+3):
                for col in range(c, c+3):
                    if board[row][col] == 0:
                        for digit in candidates[row][col]:
                            digit_positions[digit].append((row,col))

            for digit in range(1,10):
                cells = digit_positions[digit]

                # Rows
                if len(cells) == 2 and cells[0][0] == cells[1][0]:
                    target_row = cells[0][0]
                    print(f'Pointing pair of {digit}s in ROW {target_row}')
                    for x in range(9):
                        if not (c <= x < c+3) and board[target_row][x] == 0:
                            before = candidates[target_row][x].copy()
                            candidates[target_row][x].discard(digit)

                            if candidates[target_row][x] != before:
                                return True

                if len(cells) == 3 and cells[0][0] == cells[1][0] == cells[2][0]:
                    target_row = cells[0][0]
                    print(f'Pointing triple of {digit}s in ROW {target_row}')
                    for x in range(9):
                        if not (c <= x < c+3) and board[target_row][x] == 0:
                            before = candidates[target_row][x].copy()
                            candidates[target_row][x].discard(digit)

                            if candidates[target_row][x] != before:
                                return True

                # Columns
                if len(cells) == 2 and cells[0][1] == cells[1][1]:
                    target_col = cells[0][1]
                    print(f'Pointing pair of {digit}s in COL {target_col}')
                    for x in range(9):
                        if not (r <= x < r+3) and board[x][target_col] == 0:
                            before = candidates[x][target_col].copy()
                            candidates[x][target_col].discard(digit)

                            if candidates[x][target_col] != before:
                                return True

                if len(cells) == 3 and cells[0][1] == cells[1][1] == cells[2][1]:
                    target_col = cells[0][1]
                    print(f'Pointing triple of {digit}s in COL {target_col}')
                    for x in range(9):
                        if not (r <= x < r+3) and board[x][target_col] == 0:
                            before = candidates[x][target_col].copy()
                            candidates[x][target_col].discard(digit)

                            if candidates[x][target_col] != before:
                                return True

    return False

draw_board(board)

. 8 4 | 5 2 . | . . . 
. . . | . . 7 | . . . 
3 . 6 | 9 . 8 | . . . 
---------------------
2 . . | 6 . . | 8 . . 
. 6 5 | . . . | 7 4 . 
. . 8 | . . 5 | . . 2 
---------------------
. . . | 1 . 6 | 5 . 8 
. . . | 2 . . | . . . 
. . . | . 8 9 | 2 3 . 


In [224]:
# Box-Line Reduction

def box_line_reduction_rows(board, candidates):
    for r in range(9):
        digit_positions = {digit: [] for digit in range(1,10)}
        for c in range(9):
            if board[r][c] == 0:
                for digit in candidates[r][c]:
                    digit_positions[digit].append((r,c))

        for digit in range (1,10):
            cells = digit_positions[digit]

            if len(cells) in {2,3}:
                possible_cell_cols = {cells[x][1] for x in range(len(cells))}

                if possible_cell_cols.issubset({0,1,2}) or possible_cell_cols.issubset({3,4,5}) or possible_cell_cols.issubset({6,7,8}):
                    block_row_start = (r // 3) * 3
                    block_col_start = (cells[0][1] // 3) * 3
                    for row in range(block_row_start, block_row_start + 3):
                        for col in range(block_col_start, block_col_start + 3):
                            if row != r and board[row][col] == 0:
                                before = candidates[row][col].copy()
                                candidates[row][col].discard(digit)

                                if candidates[row][col] != before:
                                    print(f'Found box/line reduction for {digit}s in ROW {r}, removed from BOX {block_row_start},{block_col_start} candidates')
                                    return True
    return False

def box_line_reduction_cols(board, candidates):
    for c in range(9):
        digit_positions = {digit: [] for digit in range(1,10)}
        for r in range(9):
            if board[r][c] == 0:
                for digit in candidates[r][c]:
                    digit_positions[digit].append((r,c))

        for digit in range (1,10):
            cells = digit_positions[digit]

            if len(cells) in {2,3}:
                possible_cell_rows = {cells[x][0] for x in range(len(cells))}

                if possible_cell_rows.issubset({0,1,2}) or possible_cell_rows.issubset({3,4,5}) or possible_cell_rows.issubset({6,7,8}):
                    block_row_start = (cells[0][0] // 3) * 3
                    block_col_start = (c // 3) * 3
                    for row in range(block_row_start, block_row_start + 3):
                        for col in range(block_col_start, block_col_start + 3):
                            if col != c and board[row][col] == 0:
                                before = candidates[row][col].copy()
                                candidates[row][col].discard(digit)

                                if candidates[row][col] != before:
                                    print(f'Found box/line reduction for {digit}s in COL {c}, removed from BOX {block_row_start},{block_col_start} candidates')
                                    return True
    return False

def box_line_reduction(board, candidates):
    if box_line_reduction_rows(board, candidates):
        return True

    if box_line_reduction_cols(board, candidates):
        return True

    return False

In [224]:
# X-Wing

In [224]:
# Y-Wing

In [225]:
# Validation checks
def is_valid_row(board, row):
    desired_set = {1,2,3,4,5,6,7,8,9}
    digits = set()

    for cell in range(9):
        digits.add(board[row][cell])

    if digits == desired_set:
        return True
    else:
        return False

def is_valid_col(board, col):
    desired_set = {1,2,3,4,5,6,7,8,9}
    digits = set()
    for cell in range(9):
        digits.add(board[cell][col])

    if digits == desired_set:
        return True
    else:
        return False

def is_valid_block(board, row, col):
    desired_set = {1,2,3,4,5,6,7,8,9}
    digits = set()

    for r in range(row, row+3):
        for c in range(col, col+3):
            digits.add(board[r][c])

    if digits == desired_set:
        return True
    else:
        return False

def is_solved(board):
    errors = 0
    for row in range(9):
        is_valid = is_valid_row(board, row)
        if not is_valid:
            errors += 1
            print(f'Board not valid at row {row+1}')

    for col in range(9):
        is_valid = is_valid_col(board, col)
        if not is_valid:
            errors += 1
            print(f'Board not valid at col {col+1}')

    for row in range(0,9,3):
        for col in range(0,9,3):
            is_valid = is_valid_block(board, row, col)
            if not is_valid:
                errors += 1
                print(f'Board not valid at block ({row+1}, {col+1})')

    if errors == 0:
        print(f'Board is solved, great success')
    else:
        print(f'Board is obviously not solved :(')


In [226]:
def run_solver(board):
    candidates = get_all_candidates(board)
    while True:

        if naked_singles(board, candidates):
            candidates = get_all_candidates(board)
            continue

        if hidden_singles_rows(board, candidates):
            candidates = get_all_candidates(board)
            continue

        if hidden_singles_columns(board, candidates):
            candidates = get_all_candidates(board)
            continue

        if hidden_singles_boxes(board, candidates):
            candidates = get_all_candidates(board)
            continue

        if naked_pairs(board, candidates):
            continue

        if naked_triples(board, candidates):
            continue

        if hidden_pairs(board, candidates):
            continue

        if hidden_triples(board, candidates):
            continue

        if pointing_pairs_triples(board, candidates):
            continue

        if box_line_reduction(board, candidates):
            continue


        print("No more progress can be made.")
        draw_board(board)
        break

run_solver(board)


Hidden ROW single 8 placed at position (1, 7)
Hidden ROW single 2 placed at position (4, 5)
Hidden ROW single 8 placed at position (4, 3)
Hidden ROW single 8 placed at position (7, 0)
Hidden COLUMN single 6 placed at position (8, 0)
Hidden ROW single 5 placed at position (8, 1)
Hidden ROW single 5 placed at position (7, 4)
Hidden COLUMN single 5 placed at position (1, 0)
Hidden COLUMN single 6 placed at position (1, 4)
Hidden COLUMN single 2 placed at position (2, 7)
Hidden ROW single 5 placed at position (2, 8)
Hidden ROW single 7 placed at position (2, 1)
Hidden ROW single 5 placed at position (3, 7)
Naked set {1, 9} removed from (5, 0) using cells ((0, 0), (4, 0))
Naked set {1, 9} removed from (6, 0) using cells ((0, 0), (4, 0))
Pointing pair of 2s in ROW 1
Pointing triple of 6s in ROW 0
Pointing pair of 7s in ROW 0
Pointing triple of 9s in COL 4
Pointing pair of 6s in ROW 5
Pointing pair of 2s in ROW 6
Pointing triple of 6s in ROW 7
No more progress can be made.
. 8 4 | 5 2 . | . .

Board is solved, great success
